# 01 — Data Preprocessing
Converts raw images to JPEG, writes a nested processed image dataset, and builds `img_labels.csv` from filenames.
Filename convention: `LOCATION_TOD_WEATHER_N.JPG` (e.g. `LOCUSTWALK_daytime_sunny_1.JPG`)

In [1]:
print("hello world")

hello world


In [2]:
# ── Colab setup ──────────────────────────────────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q uv
    !uv pip install --system pillow pillow-heif pandas
    from google.colab import drive
    drive.mount('/content/drive')

In [3]:
# ── Paths — edit these ───────────────────────────────────────────────────────
if IN_COLAB:
    IMG_FOLDER   = '/content/drive/My Drive/CIS_5190_group_project/Images'
    OUTPUT_DIR   = '/content/drive/My Drive/CIS_5190_group_project/processedImages'
    LABELS_PATH  = '/content/drive/My Drive/CIS_5190_group_project/img_labels.csv'
else:
    IMG_FOLDER   = '../data/Images'
    OUTPUT_DIR   = '../data/processed'
    LABELS_PATH  = '../data/img_labels.csv'

In [5]:
import os, re
import pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError
from pillow_heif import register_heif_opener

register_heif_opener()
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set True to rebuild img_labels.csv and overwrite processed JPEGs.
REDO_PREPROCESS = False

VALID_IMAGE_SUFFIXES = {'.heic', '.heif', '.jpg', '.jpeg', '.png'}
PATTERN = r'^([^_]+)_([^_]+)_([a-zA-Z]+)'   # location_tod_weather
STEM_PATTERN = r'^([^.]+)\.'

def parse_filename(fname: str):
    m = re.match(PATTERN, fname)
    if not m:
        return None
    location, tod, weather = m.groups()
    stem = re.match(STEM_PATTERN, fname).group(1)
    return location.lower(), tod.lower(), weather.lower(), stem.lower()

parsed_files = []
badly_named = []
skipped_non_images = []
skipped_unreadable = []
for fname in sorted(os.listdir(IMG_FOLDER)):
    suffix = os.path.splitext(fname)[1].lower()
    if ':zone.identifier' in fname.lower() or suffix not in VALID_IMAGE_SUFFIXES:
        skipped_non_images.append(fname)
        continue
    parsed = parse_filename(fname)
    if parsed is None:
        badly_named.append(fname)
    else:
        parsed_files.append((fname, *parsed))

existing_df = None if REDO_PREPROCESS else (pd.read_csv(LABELS_PATH) if os.path.exists(LABELS_PATH) else None)
if REDO_PREPROCESS:
    print('REDO_PREPROCESS=True; rebuilding labels CSV and overwriting processed images.')
required_existing_cols = {'original_file_name', 'file_name', 'location_index', 'condition_dir'}
if existing_df is not None and not required_existing_cols.issubset(existing_df.columns):
    print('Existing labels CSV uses the old flat format; rebuilding with nested processed paths.')
    existing_df = None
already_done_col = 'original_file_name' if existing_df is not None and 'original_file_name' in existing_df.columns else 'file_name'
already_done = set(existing_df[already_done_col].astype(str).str.lower().values) if existing_df is not None else set()

location_to_index = {}
if existing_df is not None and {'location', 'location_index'}.issubset(existing_df.columns):
    location_to_index = dict(
        (str(location).lower(), int(index))
        for location, index in existing_df[['location', 'location_index']].drop_duplicates().values
    )
next_index = max(location_to_index.values(), default=-1) + 1
for location in sorted({row[1] for row in parsed_files}):
    if location not in location_to_index:
        location_to_index[location] = next_index
        next_index += 1

schema = {
    'original_file_name': [],
    'file_name': [],
    'location': [],
    'location_index': [],
    'time_of_day': [],
    'weather': [],
    'condition_dir': [],
}

def process_image(in_path: str, out_path: str):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    try:
        img = ImageOps.exif_transpose(Image.open(in_path)).convert('RGB')
    except (UnidentifiedImageError, OSError) as exc:
        return str(exc)
    img.save(out_path, 'JPEG', quality=95)
    return None

for fname, location, tod, weather, stem in parsed_files:
    if fname.lower() in already_done:
        continue

    in_path  = os.path.join(IMG_FOLDER, fname)
    location_index = location_to_index[location]
    condition_dir = f'{tod}_{weather}'
    out_name = (stem + '.jpg').lower()
    rel_path = os.path.join(str(location_index), condition_dir, out_name)
    out_path = os.path.join(OUTPUT_DIR, rel_path)

    error = process_image(in_path, out_path)
    if error is not None:
        skipped_unreadable.append((fname, error))
        continue

    schema['original_file_name'].append(fname.lower())
    schema['file_name'].append(rel_path.lower())
    schema['location'].append(location.lower())
    schema['location_index'].append(location_index)
    schema['time_of_day'].append(tod)
    schema['weather'].append(weather)
    schema['condition_dir'].append(condition_dir.lower())

new_df = pd.DataFrame(schema)
final_df = pd.concat([existing_df, new_df], ignore_index=True) if existing_df is not None else new_df
final_df.to_csv(LABELS_PATH, index=False)
print(f'Saved {len(final_df)} rows to {LABELS_PATH}')
if badly_named:
    print(f'\nFiles with bad names ({len(badly_named)}):')
    for f in badly_named: print(' ', f)
if skipped_non_images:
    print(f'\nSkipped non-image sidecar/files ({len(skipped_non_images)}):')
    for f in skipped_non_images[:20]: print(' ', f)
if skipped_unreadable:
    print(f'\nSkipped unreadable image files ({len(skipped_unreadable)}):')
    for f, err in skipped_unreadable[:20]: print(' ', f, '-', err)

Saved 200 rows to ../data/img_labels.csv

Skipped non-image sidecar/files (200):
  34th_night_clear.JPG:Zone.Identifier
  34th_sunset_cloudy.JPG:Zone.Identifier
  34th_sunset_cloudy_2.JPG:Zone.Identifier
  CG_SUNSET_CLEAR.HEIC:Zone.Identifier
  CG_SUNSET_CLEAR_2.HEIC:Zone.Identifier
  CG_SUNSET_CLEAR_3.HEIC:Zone.Identifier
  GREGORY_DAY_CLEAR_1 (1).HEIC:Zone.Identifier
  GREGORY_DAY_CLEAR_1.HEIC:Zone.Identifier
  GREGORY_DAY_CLEAR_2 (1).HEIC:Zone.Identifier
  GREGORY_DAY_CLEAR_2.HEIC:Zone.Identifier
  GREGORY_DAY_CLEAR_3.HEIC:Zone.Identifier
  GREGORY_DAY_CLEAR_4.HEIC:Zone.Identifier
  GREGORY_DAY_CLEAR_5.HEIC:Zone.Identifier
  GREGORY_DAY_CLOUDY(1).HEIC:Zone.Identifier
  GREGORY_DAY_CLOUDY.HEIC:Zone.Identifier
  GREGORY_DAY_CLOUDY_3.HEIC:Zone.Identifier
  GREGORY_DAY_CLOUDY_4.HEIC:Zone.Identifier
  GREGORY_NIGHT_CLOUDY.HEIC:Zone.Identifier
  GREGORY_NIGHT_CLOUDY_2.HEIC:Zone.Identifier
  GREGORY_NIGHT_CLOUDY_3.HEIC:Zone.Identifier


In [6]:
# Quick sanity check
df = pd.read_csv(LABELS_PATH)
print(df.groupby(['time_of_day', 'weather']).size().unstack(fill_value=0))

weather      clear  cloudy  rain  rainy
time_of_day                            
afternoon        8       0     0      0
day             25      18     0     26
morning         11       0     0      0
night           29      13     1      0
sunrise          5       0     0      0
sunset          34      30     0      0
